### Advanced Prompting Techniques & Best Practices

## 0. Local Infrastructure Setup

To ensure absolute compliance with enterprise data privacy standards and eliminate execution costs, this entire curriculum operates locally using **Ollama** running an optimized `llama3` instance. Run the following cells to initialize your local environment.

In [1]:
# Step 1: Install system compression tools and dependencies
!sudo apt-get update && sudo apt-get install -y zstd

Hit:1 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:2 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:4 https://cli.github.com/packages stable InRelease
Hit:5 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:8 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:10 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading packag

In [2]:
# Step 2: Download and install the core Ollama environment binary
!curl -fsSL https://ollama.com/install.sh | sh

>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [3]:
# Step 3: Spin up the Ollama background host process
import os, subprocess, time
os.environ['OLLAMA_HOST'] = '0.0.0.0:11434'
subprocess.Popen(["ollama", "serve"], stdout=subprocess.PIPE, stderr=subprocess.PIPE)
time.sleep(10)  # Verify backend execution completes system setup
print("Ollama engine successfully mounted on: http://localhost:11434")

Ollama engine successfully mounted on: http://localhost:11434


In [4]:
# Step 4: Pull down the official local 'llama3' execution engine weights
!ollama pull llama3

In [5]:
# Step 5: Establish runtime interfaces & parameterized execution wrapper
import requests
import json

OLLAMA_API_URL = "http://localhost:11434/api/generate"

def call_ollama(prompt, system_prompt=None, model="llama3", temperature=0.7, stream=False):
    """
    Encapsulates raw HTTP infrastructure interactions into an atomic, variable-controlled interface.
    Allows strict separation of system intent instructions from raw operational runtime context inputs.
    """
    # Handle incoming system configuration contexts using standard parameter patterns
    combined_prompt = f"System Context:\n{system_prompt}\n\nUser Task:\n{prompt}" if system_prompt else prompt

    payload = {
        "model": model,
        "prompt": combined_prompt,
        "stream": stream,
        "options": {
            "temperature": temperature
        }
    }

    try:
        resp = requests.post(OLLAMA_API_URL, json=payload)
        if resp.status_code != 200:
            raise RuntimeError(f"Execution gateway error: {resp.status_code} - {resp.text}")

        if stream:
            for line in resp.text.splitlines():
                if not line.strip(): continue
                data_chunk = json.loads(line)
                print(data_chunk.get("response", ""), end="")
            print("\n")
            return None
        else:
            return resp.json().get("response", "")
    except Exception as e:
        return f"System Connection Failure Encountered: {str(e)}"

print("Execution environment fully functional. System status: READY.")

Execution environment fully functional. System status: READY.


## Module 1: Prompt Debugging & The T-R-A-C-E Framework

Prompt engineering is fundamentally an engineering process. When prompts generate unintended, wrong, or malformed responses, we do not guess; we employ systematic, variable-isolated testing.

> ### The T-R-A-C-E Debugging Protocol
> 1. **T - Trigger**: Isolate what went wrong. Explicitly identify expected vs. generated results.
> 2. **R - Root Cause**: Diagnose the failure node (e.g., Ambiguity, Missing Context, Format Bugs, Scope Creep).
> 3. **A - Adjust**: Mutate exactly **one** variable component of the prompt at a time.
> 4. **C - Check**: Execute the mutation under deterministic hyperparameter conditions.
> 5. **E - Evaluate**: If performance standard is met, lock and version the prompt string. Otherwise, loop back.

Let's look at this framework in action by reviewing a standard prompt bug scenario below.

In [6]:
# --- THE AMBIGUITY BUG EXAMPLE ---
broken_prompt = """
Tell me about machine learning and stuff related to it.
"""

print("--- CRITICAL TRIGGER LOG: EXECUTION WITHOUT TRACE DEBUGGING ---")
print(call_ollama(broken_prompt, temperature=0.7))

--- CRITICAL TRIGGER LOG: EXECUTION WITHOUT TRACE DEBUGGING ---
Machine learning! It's a fascinating field that has revolutionized the way we live, work, and interact with technology. Here's an overview of machine learning and some related topics:

**What is Machine Learning?**

Machine learning is a subfield of artificial intelligence (AI) that involves training algorithms to make predictions or decisions based on data, without being explicitly programmed. The algorithm learns from experience, adjusting its behavior as it encounters new data.

**How does Machine Learning work?**

1. **Data Collection**: Gather a large dataset relevant to the problem you're trying to solve.
2. **Model Training**: Train an algorithm (e.g., neural network, decision tree) on the collected data to learn patterns and relationships.
3. **Model Evaluation**: Test the trained model on a separate dataset to evaluate its performance.
4. **Deployment**: Use the trained model in a production environment to make pr

### Applying TRACE Refinement

Our execution log demonstrates a classic **Ambiguity and Scope Creep Bug**. The model drops into essay-writing generation modes because the prompt text missing vital structural guidelines.

Let's trace modify it by explicitly locking down its **Role**, **Audience Context**, **Scope boundaries**, and **Format specifications**.

In [7]:
# --- TRACE DEBUGGED PROMPT ENHANCEMENT ---
fixed_prompt = """
Act as a machine learning tutor for beginners.
Explain supervised learning in exactly 5 bullet points.
Keep each bullet under 2 sentences.
"""

print("--- SYSTEM CHECK LOG: TRACE ADJUSTMENT COMPLETE ---")
print(call_ollama(fixed_prompt, temperature=0.3))

--- SYSTEM CHECK LOG: TRACE ADJUSTMENT COMPLETE ---
I'd be happy to help you learn about supervised learning. Here are five key points:

• **Definition:** Supervised learning is a type of machine learning where the algorithm learns from labeled data, meaning each example has a corresponding target or response variable.

• **Goal:** The goal of supervised learning is to train a model that can predict the target variable based on the input features, so it can make accurate predictions on new, unseen data.

• **Example:** A classic example of supervised learning is image classification, where the algorithm learns to classify images as either "dog" or "cat" based on labeled training data (e.g., a picture of a dog with the label "dog").

• **Types of Supervised Learning:** There are two main types of supervised learning: regression and classification. Regression involves predicting a continuous value, while classification involves predicting a categorical label.

• **Evaluation:** To evalua

---

## Module 2: Prompt Guardrails

Guardrails are system configurations embedded explicitly into the user context or system-level configuration window. Their core goal is to establish bounding criteria that keep AI behavior highly predictable, appropriate, and strictly aligned with enterprise constraints under edge cases.

### Types of Guardrails
1. **Scope Fences**: Forcing the model to remain bound to a chosen dataset or field topic.
2. **Format Locks**: Restricting the vocabulary or template wrapper structure (e.g., standard valid JSON output parsing).
3. **Behavior & Tone Fences**: Restricting age range, grade vocabulary levels, or corporate voice requirements.

In [8]:
# --- ENFORCING STRUCTURAL FORMAT BLOCKS VIA COMPONENT PATTERNS ---

json_enforcement_guardrail = """
You are a raw system parser data extractor.
Always respond in valid JSON. Never include markdown wrappers, styling blockticks, or prose.

Required Output Schema:
{
  "detected_language": string,
  "sentiment_score": float between -1.0 and 1.0,
  "entities_extracted": list of strings
}
"""

target_input = "I absolutely hate when the new application dashboard randomly crashes during production drops! It's so frustrating."

print("--- STRUCTURAL COMPLIANCE VERIFICATION EXECUTION ---")
print(call_ollama(prompt=target_input, system_prompt=json_enforcement_guardrail, temperature=0.0))

--- STRUCTURAL COMPLIANCE VERIFICATION EXECUTION ---
{
  "detected_language": "en",
  "sentiment_score": -0.8,
  "entities_extracted": ["application", "dashboard", "production"]


## Module 3: Hands-On Case Study — Debugging the TechHelp Chatbot

### Business Operational Context
An enterprise software engineering company has deployed an LLM-powered virtual assistant solution named **TechHelp** to automate user software support requests. However, system deployment telemetry shows major regressions that damage customer trust.

We have isolated three critical systemic prompt bugs that we must systematically analyze, reproduce, debug, and fix using advanced guardrail architecture patterns.

### Reproducing the Failures (Baseline System Prompt)

In [9]:
baseline_system_prompt = """
You are TechHelp, a customer support assistant. Answer user questions helpfully.
"""

In [10]:
# BUG 1 REPRODUCTION: Hallucination Bug
# Context: The product line only spans versions 1.0 to 2.5. Version 3 does not exist.
hallucination_test_input = "What are the system requirements for TechHelp v3?"

print("--- BUG 1 EXECUTION OUTPUT ---")
print(call_ollama(prompt=hallucination_test_input, system_prompt=baseline_system_prompt, temperature=0.7))

--- BUG 1 EXECUTION OUTPUT ---
I'm happy to help you with that!

TechHelp v3 is designed to be compatible with a wide range of devices and operating systems. Here are the minimum system requirements:

**Operating System:**

* Windows 10 (64-bit) or later
* macOS High Sierra (or later) or equivalent Linux distribution

**Processor:**

* Dual-core processor (Intel Core i3 or AMD equivalent)
* Clock speed: 2.5 GHz or higher

**Memory (RAM):**

* 8 GB RAM (16 GB or more recommended)

**Storage:**

* Minimum 20 GB available disk space (30 GB or more recommended)

**Browser:**

* Google Chrome (latest version) or Mozilla Firefox (latest version)
* Microsoft Edge (latest version) is also supported

**Internet Connection:**

* Stable internet connection with a minimum speed of 1 Mbps (upstream) and 0.5 Mbps (downstream)

Please note that these are the minimum system requirements, and having a more powerful device will ensure a smoother experience.

If you have any further questions or concerns

In [11]:
# BUG 2 REPRODUCTION: Scope Violation Bug
# Context: The support system is processing general internet questions completely unrelated to product lines.
scope_test_input = "What's the weather like in New York right now?"

print("--- BUG 2 EXECUTION OUTPUT ---")
print(call_ollama(prompt=scope_test_input, system_prompt=baseline_system_prompt, temperature=0.7))

--- BUG 2 EXECUTION OUTPUT ---
I'm happy to help!

To give you an update on the current weather in New York, I'd recommend checking out a reliable weather website or app. One option is accuweather.com, which provides up-to-the-minute forecasts and conditions for various cities around the world.

As of my knowledge cutoff (which might be slightly outdated), according to AccuWeather, it was partly cloudy with a high of 68°F (20°C) and a low of 54°F (12°C) in New York City. However, please note that weather conditions can change quickly, so I'd encourage you to check the latest updates for the most accurate information.

If you need any further assistance or have another question, feel free to ask!


In [12]:
# BUG 3 REPRODUCTION: Format Failure Bug
# Context: The assistant outputs massive, dense run-on text walls instead of digestible, actionable steps.
format_test_input = "List the top 5 features of TechHelp."

print("--- BUG 3 EXECUTION OUTPUT ---")
print(call_ollama(prompt=format_test_input, system_prompt=baseline_system_prompt, temperature=0.7))

--- BUG 3 EXECUTION OUTPUT ---
Hello there! I'm TechHelp, your friendly customer support assistant. I'd be happy to introduce you to my top 5 features that make me super helpful!

**1.** **Multilingual Support**: I speak multiple languages, including English, Spanish, French, German, Italian, and more! Don't worry if language barriers are holding you back; I'll communicate with you in your native tongue.

**2.** **24/7 Availability**: Need help at 3 AM? No problem! I'm always available to assist you, whether it's a weekend or a holiday. Just type away, and I'll be there in no time!

**3.** **In-Depth Knowledge Base**: My vast knowledge base is regularly updated with the latest information on various products, services, and topics. You can search for answers to common questions, and if I don't have it, I'll find out who does!

**4.** **Personalized Support**: Every user is unique, and so is your issue! I take the time to understand your specific problem, provide customized solutions, an

---

### Implementing Production-Grade Structural Guardrails

We will now resolve all three bugs systematically by crafting an advanced, section-isolated system prompt. This prompt combines topic boundaries, uncertainty routing rules, and strict formatting templates into a single high-performance deployment block.

In [13]:
production_system_prompt = """
You are TechHelp, the official customer support assistant for TechHelp Software.
Your knowledge covers TechHelp versions 1.0 through 2.5.

━━━ SCOPE GUARDRAIL ━━━
ONLY answer questions about TechHelp software.
For off-topic questions (such as general knowledge, weather, news, or unrelated software), say:
"I specialize in TechHelp support. For [topic], please use a general resource. How can I help you with TechHelp today?"

━━━ HALLUCINATION GUARDRAIL ━━━
If you are uncertain about a fact, version, or specification, or if it is not explicitly present within your version history context (v1.0-v2.5):
ALWAYS say: "I want to make sure I give you accurate info. Please verify at docs.techhelp.com"
NEVER invent version numbers, specifications, dates, or feature details.

━━━ FORMAT GUARDRAIL ━━━
- For lists: Use numbered format. One item per line. Bold the key term.
- For troubleshooting: Use a numbered step-by-step process.
- For comparisons: Use a two-column structure.
- Always end with: "Does this help? Is there anything else about TechHelp I can clarify?"

━━━ TONE ━━━
Professional, friendly, concise. Maximum 150 words per response unless code is needed.
"""

### Testing and Verifying our Production Guardrails

In [14]:
print("--- TEST 1 RESULT (VERIFYING ANTI-HALLUCINATION RESOLUTION) ---")
print(call_ollama(prompt=hallucination_test_input, system_prompt=production_system_prompt, temperature=0.2))

--- TEST 1 RESULT (VERIFYING ANTI-HALLUCINATION RESOLUTION) ---
I specialize in TechHelp support. For information on TechHelp v3, please use a general resource. How can I help you with TechHelp today?

As my knowledge only covers versions 1.0 through 2.5, I'm not familiar with the system requirements for TechHelp v3. If you're looking to upgrade or install a newer version, I recommend checking the official documentation at docs.techhelp.com.

Does this help? Is there anything else about TechHelp I can clarify?


In [15]:
print("--- TEST 2 RESULT (VERIFYING TOPIC SCOPE CONTROL) ---")
print(call_ollama(prompt=scope_test_input, system_prompt=production_system_prompt, temperature=0.2))

--- TEST 2 RESULT (VERIFYING TOPIC SCOPE CONTROL) ---
I specialize in TechHelp support. For [topic], please use a general resource. How can I help you with TechHelp today?


In [16]:
print("--- TEST 3 RESULT (VERIFYING FORMAT ENFORCEMENT COMPLIANCE) ---")
print(call_ollama(prompt=format_test_input, system_prompt=production_system_prompt, temperature=0.2))

--- TEST 3 RESULT (VERIFYING FORMAT ENFORCEMENT COMPLIANCE) ---
I'd be happy to help you with that!

**Top 5 Features of TechHelp:**

1. **Automated Backup**: Schedule automatic backups of your data to ensure it's safe and secure.
2. **Customizable Dashboards**: Tailor your workspace to fit your needs by rearranging widgets, adding new ones, or hiding unnecessary ones.
3. **Real-time Collaboration**: Work seamlessly with teammates in real-time, making it easy to share files, ideas, and feedback.
4. **Task Prioritization**: Organize tasks using our intuitive prioritization system, ensuring you stay focused on the most important projects.
5. **Intelligent Search**: Quickly find what you need with our advanced search functionality that learns your search patterns and preferences.

Does this help? Is there anything else about TechHelp I can clarify?


## Module 4: Building an Automated Evaluation Pipeline

System validation cannot depend entirely on manual developer spot-checks. If a prompt optimization modifies an instruction block, we must ensure it doesn't cause formatting or accuracy regressions elsewhere.

We will build a programmatic assertion evaluation suite to track baseline metrics automatically across multiple functional criteria.

In [17]:
# 1. Define our evaluation test suite datasets
eval_test_suite = [
    {
        "id": "CASE_01_HALLUCINATION",
        "input": "What are the hardware specifications needed to run TechHelp version 3.0 smoothly?",
        "eval_type": "anti_hallucination"
    },
    {
        "id": "CASE_02_SCOPE",
        "input": "Can you provide me a simple recipe for baking chocolate chip cookies?",
        "eval_type": "scope_check"
    },
    {
        "id": "CASE_03_FORMAT",
        "input": "List the primary 5 features found in the core TechHelp architecture system layout.",
        "eval_type": "format_check"
    }
 ]

# 2. Establish deterministic validation rule sets
def evaluate_response(case_type, output_text):
    """
    Executes deterministic logic-based rule evaluations against the generated response text.
    Returns 1.0 for absolute compliance, 0.0 for rule breakage.
    """
    normalized_text = output_text.lower()

    if case_type == "anti_hallucination":
        # Must successfully route to documentation safety links rather than asserting false facts
        if "docs.techhelp.com" in normalized_text and not ("v3.0" in normalized_text and "require" in normalized_text):
            return 1.0
        return 0.0

    elif case_type == "scope_check":
        # Must trigger redirect notices without answering the out-of-scope question
        if "specialize in techhelp" in normalized_text or "general resource" in normalized_text:
            if "sugar" not in normalized_text and "bake" not in normalized_text:
                return 1.0
        return 0.0

    elif case_type == "format_check":
        # Must enforce readable structured lists with clear numbers
        lines = normalized_text.split("\n")
        numbered_lines = [l for l in lines if any(char.isdigit() for char in l.split('. ')[0] if char.isdigit())]
        if len(numbered_lines) >= 3:
            return 1.0
        return 0.0

    return 0.0

In [18]:
# 3. Run the automated execution pipeline against candidate system prompt setups
def execute_eval_pipeline(candidate_system_prompt):
    print(f"| {'Test ID':<22} | {'Evaluation Strategy':<22} | {'Pipeline Status':<15} |")
    print("|" + "-"*24 + "|" + "-"*24 + "|" + "-"*17 + "|")

    total_score = 0.0

    for test_case in eval_test_suite:
        response = call_ollama(
            prompt=test_case["input"],
            system_prompt=candidate_system_prompt,
            temperature=0.0  # Zero out creativity variables to lock down testing determinism
        )

        score = evaluate_response(test_case["eval_type"], response)
        total_score += score
        status = "PASSED" if score == 1.0 else "FAILED"

        print(f"| {test_case['id']:<22} | {test_case['eval_type']:<22} | {status:<15} |")

    avg_score = total_score / len(eval_test_suite)
    print("\n" + "="*40)
    print(f"Cumulative Prompt Compliance Score: {avg_score:.1%}")
    print("="*40)

print("--- PIPELINE EXECUTION: BASELINE CHECKS ---")
execute_eval_pipeline(baseline_system_prompt)

print("\n\n--- PIPELINE EXECUTION: PRODUCTION GUARDRAIL SYSTEM CHECKS ---")
execute_eval_pipeline(production_system_prompt)

--- PIPELINE EXECUTION: BASELINE CHECKS ---
| Test ID                | Evaluation Strategy    | Pipeline Status |
|------------------------|------------------------|-----------------|
| CASE_01_HALLUCINATION  | anti_hallucination     | FAILED          |
| CASE_02_SCOPE          | scope_check            | FAILED          |
| CASE_03_FORMAT         | format_check           | PASSED          |

Cumulative Prompt Compliance Score: 33.3%


--- PIPELINE EXECUTION: PRODUCTION GUARDRAIL SYSTEM CHECKS ---
| Test ID                | Evaluation Strategy    | Pipeline Status |
|------------------------|------------------------|-----------------|
| CASE_01_HALLUCINATION  | anti_hallucination     | FAILED          |
| CASE_02_SCOPE          | scope_check            | PASSED          |
| CASE_03_FORMAT         | format_check           | PASSED          |

Cumulative Prompt Compliance Score: 66.7%


---

## Module 5: Advanced Response Optimization

When dealing with complex, multi-step analytical challenges (such as log data triage, dependency matrix evaluation, or strict reasoning tasks), simple instruction lines can fail. We solve this using advanced response optimization primitives.

### Optimization Modalities
* **Few-Shot Prompting**: Providing inline demonstrations of the target pattern.
* **Chain-of-Thought (CoT)**: Instructing the model to map out its logical calculations explicitly before outputting its final conclusion.

### Chain-of-Thought (CoT) Application Execution Model

In [19]:
# Without Chain-of-Thought optimization, models frequently struggle with mathematical calculations or logic constraints.
reasoning_task = """
A server farm hosting TechHelp instances contains 3 server racks. Each rack has 12 blade nodes.
Racks 1 and 2 run at 80% maximum compute capacity capacity.
Rack 3 is running completely dark at 0% efficiency capacity.
What is the total active operational compute node capacity count across the facility currently?
Provide only the absolute final total number value response.
"""

print("--- OPTIMIZATION CHECK: BASELINE TASK EXECUTION ---")
print(call_ollama(reasoning_task, temperature=0.0))

--- OPTIMIZATION CHECK: BASELINE TASK EXECUTION ---
2 racks are running at 80% capacity, so:

Rack 1: 12 nodes x 80% = 9.6 nodes (round down to 9)
Rack 2: 12 nodes x 80% = 9.6 nodes (round down to 9)

Total active nodes in Racks 1 and 2: 9 + 9 = 18

Rack 3 is running at 0%, so it has no active nodes.

Total active operational compute node capacity count across the facility: **18**


In [20]:
# Upgrading the logic flow using an embedded Chain-of-Thought optimization trigger pattern
optimized_reasoning_task = reasoning_task + "\nReason step-by-step through the calculation parameters. Outline your work before outputting the final total."

print("--- OPTIMIZATION CHECK: CHAIN-OF-THOUGHT UPGRADE ---")
print(call_ollama(optimized_reasoning_task, temperature=0.0))

--- OPTIMIZATION CHECK: CHAIN-OF-THOUGHT UPGRADE ---
Let's break down the problem step by step:

1. Each rack has 12 blade nodes.
2. Racks 1 and 2 are running at 80% maximum compute capacity, which means they have 0.8 x 12 = 9.6 active nodes per rack (we'll round this to 9.6 for simplicity).
3. Rack 3 is completely dark at 0% efficiency capacity, so it has 0 active nodes.

Now, let's calculate the total active operational compute node capacity count across the facility:

Rack 1: 9.6 nodes
Rack 2: 9.6 nodes
Rack 3: 0 nodes

Total active nodes = 9.6 + 9.6 + 0 = 19.2

Since we can't have a fraction of a node, we'll round this down to the nearest whole number:

Final total: **19**
